In [7]:
# Majority Voting

import json

file_train = "../data/country_select_result.json"
with open(file_train, "r", encoding='latin-1') as f:
    train_json = json.load(f)

import json
import random
random.seed(42)
import math
import numpy
import matplotlib.pyplot as plt

def get_country_dict():
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', 
            'china': 'chn',     'united states': 'usa',   'russia': 'rus', 'united kingdom': 'gbr', 'france': 'fra',
            'germany': 'deu',   'korea': 'kor',     'japan': 'jpn',  'india': 'ind',     'canada': 'can', 
            'italy': 'ita',     'australia': 'aus', 'spain': 'esp',  'argentina': 'arg', 'brazil': 'bra',
            'indonesia': 'idn', 'mexico': 'mex',    'south africa': 'zaf'}

def get_mideast_country_dict():
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', }

def find_most_frequent_string(strings, max_number):
    # 使用字典统计每个字符串出现的次数
    frequency = {}
    for s in strings:
        frequency[s] = frequency.get(s, 0) + 1
    
    # 找出最高出现次数
    max_frequency = max(frequency.values())
    
    # 找出所有出现次数最多的字符串
    most_frequent = [s for s, count in frequency.items() if count == max_frequency]
    
    if len(most_frequent) == 1:
        max_number = max_number + 1

    # 随机选择一个最频繁的字符串
    return random.choice(most_frequent), max_number

all_country_dict = get_country_dict()
mideast_country_dict = get_mideast_country_dict()

with open("/mnt/ssd1/hxli/LLM_Event/ThinkTank-ME/POLECAT-FOR-ME/4_routing_dataset/country_select_max100_min20/country_forecast_results_test.json", "r", encoding='latin-1') as f:
    all_json = json.load(f)

beam_size_i = 1
country_select_beam_size_i = 21 # from 1 to 35, 21 is the best one

# max_number 指最终回答中出现次数最多的选择只有一个的数量
max_number = 0
right_i = 0

all_number_country = {}
right_number_country = {}
for index_2i, content_2i in mideast_country_dict.items():
    all_number_country[index_2i] = 0
    right_number_country[index_2i] = 0

all_len = len(all_json)

for index_3i in range(all_len):

    target_i = all_json[str(index_3i)]["target"]
    target_country_code_i = all_json[str(index_3i)]["country_code"]
    target_country_name_i = all_json[str(index_3i)]["country_name"]

    strings_list = []
    w_list = []

    for index_3ii, content_3ii in enumerate(train_json[str(index_3i)]["output"][:country_select_beam_size_i]):
        country_name_i = content_3ii.strip().split("<|")[0].lower()
        ###
        # print(country_id_i)
        output_1_i = all_json[str(index_3i)]["output"][country_name_i]
        strings_list.append(output_1_i)
    
        w_1_i = all_json[str(index_3i)]["scores"][country_name_i]
        w_list.append(math.exp(w_1_i))

    output_i, max_number = find_most_frequent_string(strings_list, max_number)

    if target_i == output_i:
        right_i = right_i + 1
    for index_3ii, content_3ii in mideast_country_dict.items():
        if target_country_code_i == content_3ii:
            all_number_country[index_3ii] = all_number_country[index_3ii] + 1
            if target_i == output_i:
                right_number_country[index_3ii] = right_number_country[index_3ii] + 1

Macro_average = []
Micro_average = right_i/all_len
print(f"Mi: {right_i/all_len:.2%} ({right_i}/{all_len})", end=" ")
print()

for index_4i, content_4i in mideast_country_dict.items():
    Macro_average.append(right_number_country[index_4i]/all_number_country[index_4i])
    print(f"{index_4i}: {right_number_country[index_4i]/all_number_country[index_4i]} ({right_number_country[index_4i]}/{all_number_country[index_4i]})")
    
Macro_average = numpy.mean(Macro_average)
print(f"Ma: {Macro_average:.2%}", end="  ")


Mi: 23.94% (1821/7605) 
iran: 0.17349726775956284 (127/732)
israel: 0.18032786885245902 (407/2257)
egypt: 0.12844036697247707 (84/654)
saudi arabia: 0.18478260869565216 (17/92)
turkey: 0.29714285714285715 (52/175)
iraq: 0.13537117903930132 (31/229)
yemen: 0.5067729083665339 (636/1255)
syria: 0.22626788036410922 (174/769)
jordan: 0.17834394904458598 (28/157)
united arab emirates: 0.23809523809523808 (10/42)
lebanon: 0.2686230248306998 (119/443)
oman: 0.25 (7/28)
kuwait: 0.0 (0/6)
qatar: 0.112 (42/375)
bahrain: 0.7352941176470589 (25/34)
cyprus: 0.3333333333333333 (3/9)
palestine: 0.16954022988505746 (59/348)
Ma: 24.22%  

In [6]:
# Vanilla Best of N

import json

# file_train = "../data/test_CS_Trainbeam_24.json"
file_train = "../data/country_select_result.json"
with open(file_train, "r", encoding='latin-1') as f:
    train_json = json.load(f)


import json
import random
random.seed(42)
import math
import numpy
import matplotlib.pyplot as plt

def get_country_dict():
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', 
            'china': 'chn',     'united states': 'usa',   'russia': 'rus', 'united kingdom': 'gbr', 'france': 'fra',
            'germany': 'deu',   'korea': 'kor',     'japan': 'jpn',  'india': 'ind',     'canada': 'can', 
            'italy': 'ita',     'australia': 'aus', 'spain': 'esp',  'argentina': 'arg', 'brazil': 'bra',
            'indonesia': 'idn', 'mexico': 'mex',    'south africa': 'zaf'}

def get_mideast_country_dict():
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', }

def find_most_weighted_string(strings, weights, max_number):
    # 创建字典存储每个字符串的累积权重
    weighted_frequency = {}
    
    # 计算每个字符串的累积权重
    for s, w in zip(strings, weights):
        weighted_frequency[s] = weighted_frequency.get(s, 0) + w
    
    # 找出最高权重
    max_weight = max(weighted_frequency.values())
    
    # 找出所有权重最高的字符串
    most_weighted = [s for s, weight in weighted_frequency.items() if weight == max_weight]
    
    if len(most_weighted) == 1:
        max_number = max_number + 1

    # 随机选择一个权重最高的字符串
    return random.choice(most_weighted), max_number

all_country_dict = get_country_dict()
mideast_country_dict = get_mideast_country_dict()

with open("/mnt/ssd1/hxli/LLM_Event/ThinkTank-ME/POLECAT-FOR-ME/4_routing_dataset/country_select_max100_min20/country_forecast_results_test.json", "r", encoding='latin-1') as f:
    all_json = json.load(f)

beam_size_i = 1
country_select_beam_size_i = 18 # from 1 to 35, 18 is the best one

# max_number 指最终回答中出现次数最多的选择只有一个的数量
max_number = 0
right_i = 0

all_number_country = {}
right_number_country = {}
for index_2i, content_2i in mideast_country_dict.items():
    all_number_country[index_2i] = 0
    right_number_country[index_2i] = 0

for index_3i in range(all_len):
    target_i = all_json[str(index_3i)]["target"]
    target_country_code_i = all_json[str(index_3i)]["country_code"]
    target_country_name_i = all_json[str(index_3i)]["country_name"]

    strings_list = []
    w_list = []

    #for index_3ii, content_3ii in all_country_dict.items():
    ###
    for index_3ii, content_3ii in enumerate(train_json[str(index_3i)]["output"][:country_select_beam_size_i]):
        country_name_i = content_3ii.strip().split("<|")[0].lower()
        ###
        # print(country_id_i)
        output_1_i = all_json[str(index_3i)]["output"][country_name_i]
        strings_list.append(output_1_i)
    
        w_1_i = all_json[str(index_3i)]["scores"][country_name_i]
        w_list.append(math.exp(w_1_i))

    # output_i, max_number = find_most_weighted_string(strings_list, w_list, max_number)
    output_i = strings_list[numpy.argmax(w_list)]

    if target_i == output_i:
        right_i = right_i + 1
    for index_3ii, content_3ii in mideast_country_dict.items():
        if target_country_code_i == content_3ii:
            all_number_country[index_3ii] = all_number_country[index_3ii] + 1
            if target_i == output_i:
                right_number_country[index_3ii] = right_number_country[index_3ii] + 1

Macro_average = []
Micro_average = right_i/all_len
print(f"Mi: {right_i/all_len:.2%} ({right_i}/{all_len})", end=" ")
print()

for index_4i, content_4i in mideast_country_dict.items():
    Macro_average.append(right_number_country[index_4i]/all_number_country[index_4i])
    print(f"{index_4i}: {right_number_country[index_4i]/all_number_country[index_4i]} ({right_number_country[index_4i]}/{all_number_country[index_4i]})")
    
Macro_average = numpy.mean(Macro_average)
print(f"Ma: {Macro_average:.2%}", end="  ")


Mi: 22.81% (1735/7605) 
iran: 0.13934426229508196 (102/732)
israel: 0.17678334071776694 (399/2257)
egypt: 0.14067278287461774 (92/654)
saudi arabia: 0.1956521739130435 (18/92)
turkey: 0.2857142857142857 (50/175)
iraq: 0.11790393013100436 (27/229)
yemen: 0.5091633466135458 (639/1255)
syria: 0.19245773732119636 (148/769)
jordan: 0.1337579617834395 (21/157)
united arab emirates: 0.14285714285714285 (6/42)
lebanon: 0.24830699774266365 (110/443)
oman: 0.25 (7/28)
kuwait: 0.0 (0/6)
qatar: 0.08 (30/375)
bahrain: 0.7352941176470589 (25/34)
cyprus: 0.5555555555555556 (5/9)
palestine: 0.16091954022988506 (56/348)
Ma: 23.91%  

In [5]:
# Weighted Best of N

import json

# file_train = "../data/test_CS_Trainbeam_24.json"
file_train = "../data/country_select_result.json"
with open(file_train, "r", encoding='latin-1') as f:
    train_json = json.load(f)

import json
import random
random.seed(42)
import math
import numpy
import matplotlib.pyplot as plt

def get_country_dict():
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', 
            'china': 'chn',     'united states': 'usa',   'russia': 'rus', 'united kingdom': 'gbr', 'france': 'fra',
            'germany': 'deu',   'korea': 'kor',     'japan': 'jpn',  'india': 'ind',     'canada': 'can', 
            'italy': 'ita',     'australia': 'aus', 'spain': 'esp',  'argentina': 'arg', 'brazil': 'bra',
            'indonesia': 'idn', 'mexico': 'mex',    'south africa': 'zaf'}

def get_mideast_country_dict():
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', }

def find_most_weighted_string(strings, weights, max_number):
    # 创建字典存储每个字符串的累积权重
    weighted_frequency = {}
    
    # 计算每个字符串的累积权重
    for s, w in zip(strings, weights):
        weighted_frequency[s] = weighted_frequency.get(s, 0) + w
    
    # 找出最高权重
    max_weight = max(weighted_frequency.values())
    
    # 找出所有权重最高的字符串
    most_weighted = [s for s, weight in weighted_frequency.items() if weight == max_weight]
    
    if len(most_weighted) == 1:
        max_number = max_number + 1

    # 随机选择一个权重最高的字符串
    return random.choice(most_weighted), max_number

all_country_dict = get_country_dict()
mideast_country_dict = get_mideast_country_dict()

with open("/mnt/ssd1/hxli/LLM_Event/ThinkTank-ME/POLECAT-FOR-ME/4_routing_dataset/country_select_max100_min20/country_forecast_results_test.json", "r", encoding='latin-1') as f:
    all_json = json.load(f)

beam_size_i = 1
country_select_beam_size_i = 17 # from 1 to 35, 17 is the best one

# max_number 指最终回答中出现次数最多的选择只有一个的数量
max_number = 0
right_i = 0

all_number_country = {}
right_number_country = {}
for index_2i, content_2i in mideast_country_dict.items():
    all_number_country[index_2i] = 0
    right_number_country[index_2i] = 0

all_len = len(all_json)

for index_3i in range(all_len):
    target_i = all_json[str(index_3i)]["target"]
    target_country_code_i = all_json[str(index_3i)]["country_code"]
    target_country_name_i = all_json[str(index_3i)]["country_name"]

    strings_list = []
    w_list = []

    for index_3ii, content_3ii in enumerate(train_json[str(index_3i)]["output"][:country_select_beam_size_i]):
        country_name_i = content_3ii.strip().split("<|")[0].lower()
        ###
        # print(country_id_i)
        output_1_i = all_json[str(index_3i)]["output"][country_name_i]
        strings_list.append(output_1_i)
    
        w_1_i = all_json[str(index_3i)]["scores"][country_name_i]
        w_list.append(math.exp(w_1_i))

    output_i, max_number = find_most_weighted_string(strings_list, w_list, max_number)

    if target_i == output_i:
        right_i = right_i + 1
    for index_3ii, content_3ii in mideast_country_dict.items():
        if target_country_code_i == content_3ii:
            all_number_country[index_3ii] = all_number_country[index_3ii] + 1
            if target_i == output_i:
                right_number_country[index_3ii] = right_number_country[index_3ii] + 1

Macro_average = []
Micro_average = right_i/all_len
print(f"Mi: {right_i/all_len:.2%} ({right_i}/{all_len})", end=" ")
print()

for index_4i, content_4i in mideast_country_dict.items():
    Macro_average.append(right_number_country[index_4i]/all_number_country[index_4i])
    print(f"{index_4i}: {right_number_country[index_4i]/all_number_country[index_4i]} ({right_number_country[index_4i]}/{all_number_country[index_4i]})")
    
Macro_average = numpy.mean(Macro_average)
print(f"Ma: {Macro_average:.2%}", end="  ")



Mi: 24.58% (1869/7605) 
iran: 0.17486338797814208 (128/732)
israel: 0.18342933097031458 (414/2257)
egypt: 0.15749235474006115 (103/654)
saudi arabia: 0.20652173913043478 (19/92)
turkey: 0.2857142857142857 (50/175)
iraq: 0.14410480349344978 (33/229)
yemen: 0.5163346613545817 (648/1255)
syria: 0.2340702210663199 (180/769)
jordan: 0.17197452229299362 (27/157)
united arab emirates: 0.19047619047619047 (8/42)
lebanon: 0.2686230248306998 (119/443)
oman: 0.25 (7/28)
kuwait: 0.0 (0/6)
qatar: 0.11466666666666667 (43/375)
bahrain: 0.7352941176470589 (25/34)
cyprus: 0.4444444444444444 (4/9)
palestine: 0.1752873563218391 (61/348)
Ma: 25.02%  